# All-Lineage Combined scVI + scANVI Pipeline — v1.3.1 FINAL
**Author:** r2end | **Date:** 2026-03-19 | **Version:** v1.3.1 FINAL

**Key changes over v1.3:**
- `BATCH_KEY = "sample"` unified across all lineages (v1.3.1-1)
- `resolve_fullgene_counts_matrix()` returns CSR float32 (v1.3.1-2)
- Lineage-level log1p pre-build removed (v1.3.1-3)
- `_safe_label_column()` zero silent fallback for non-B-cell lineages (v1.3.1-4)
- Soft integer diagnostic warning retained (v1.3.1-5)
- **Built-in Tee logging**: all output mirrored to timestamped log file (v1.3.1-6)

**Notebook log:** path printed in Cell 1 output.
Run all cells sequentially. Log file survives kernel restart.

## Cell 1 — Logging Setup + Imports
> **Run this cell first.** Creates timestamped log file and mirrors all `print()` / `stderr` output to it for the rest of the session.

In [4]:
# ===== LOGGING SETUP (must be first) =====
# Tee: mirrors stdout/stderr to both notebook output and a timestamped log file.
# Works transparently -- all subsequent print() calls are captured automatically.

import os, sys

os.environ["OMP_NUM_THREADS"]      = "8"
os.environ["OPENBLAS_NUM_THREADS"] = "8"
os.environ["MKL_NUM_THREADS"]      = "8"

import matplotlib
matplotlib.use("Agg")

import gc, time, json, anndata
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import scipy.sparse as sparse
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
import torch

# ---- Tee class ----
class _Tee:
    """Write to both the original stream and a log file simultaneously."""
    def __init__(self, original, log_fh):
        self._orig   = original
        self._log    = log_fh
    def write(self, data):
        self._orig.write(data)
        self._orig.flush()
        try:
            self._log.write(data)
            self._log.flush()
        except Exception:
            pass
    def flush(self):
        self._orig.flush()
        try: self._log.flush()
        except Exception: pass
    def fileno(self):           # needed by some C-level code
        return self._orig.fileno()
    def isatty(self):
        return False

# ---- Log file path ----
_LOG_BASE = Path("/home/h2048/data/py/20260319/allcells_combined_scanvi/logs")
_LOG_BASE.mkdir(parents=True, exist_ok=True)
_TS        = datetime.now().strftime("%Y%m%d_%H%M%S")
_NB_STEM   = "allcells_combined_scanvi_20260319_v1_3_1_FINAL"
LOG_FILE   = _LOG_BASE / f"{_NB_STEM}_{_TS}.log"

_log_fh        = open(LOG_FILE, "w", encoding="utf-8", buffering=1)
sys.stdout     = _Tee(sys.__stdout__, _log_fh)
sys.stderr     = _Tee(sys.__stderr__, _log_fh)

# ---- Runtime info ----
anndata.settings.allow_write_nullable_strings = True
GPU_AVAILABLE  = torch.cuda.is_available()
_accelerator   = "gpu" if GPU_AVAILABLE else "cpu"
_devices       = 1     if GPU_AVAILABLE else "auto"
scvi.settings.seed = 42
if GPU_AVAILABLE:
    torch.cuda.manual_seed_all(42)
np.random.seed(42)
sc.settings.n_jobs    = 16
sc.settings.verbosity = 2
PIPELINE_START = time.time()

print("=" * 70)
print(f"Pipeline   : {_NB_STEM}")
print(f"Log file   : {LOG_FILE}")
print(f"Start      : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"scvi-tools : {scvi.__version__}")
print(f"scanpy     : {sc.__version__}")
print(f"Accelerator: {_accelerator}  devices={_devices}")
print(f"GPU avail  : {GPU_AVAILABLE}")
print("=" * 70)
print()
print("Monitor log in terminal:")
print(f"  tail -f {LOG_FILE}")
print()
print("Check process:")
print(f"  ps aux | grep jupyter | grep -v grep")

## Cell 2 — Configuration

In [ ]:
DATE_TAG   = "20260319"
VERSION    = "1_3_1"
OUTPUT_DIR = Path(f"/home/h2048/data/py/{DATE_TAG}/allcells_combined_scanvi")
MODEL_DIR  = OUTPUT_DIR / "models"
FIG_DIR    = OUTPUT_DIR / "figures"

for d in [OUTPUT_DIR, MODEL_DIR, FIG_DIR, _LOG_BASE]:
    d.mkdir(parents=True, exist_ok=True)

# [v1.3.1-1] All lineages use sample-level batch (unified granularity)
LINEAGE_INPUTS = {
    "epithelial": {
        "h5ad"           : Path("/home/h2048/data/py/0317/epithelial_v2_7_HOTFIX/SELF/"
                                "epithelial_scanvi_v2_7_HOTFIX_SELF_final.h5ad"),
        "label"          : "ann_level_3_subcluster",
        "name"           : "Epithelial",
        "batch_key_local": "sample",
    },
    "tcell": {
        "h5ad"           : Path("/home/h2048/data/py/0318/tnk_subcluster_retrain/"
                                "adata_tnk_scanvi_ref_retrain_v1_2.h5ad"),
        "label"          : "scanvi_label_refined",
        "name"           : "TNK",
        "batch_key_local": "sample",
    },
    "myeloid": {
        "h5ad"           : Path("/home/h2048/data/py/0209/myeloid_validation_optimized/"
                                "adata_myeloid_refined_FINAL.h5ad"),
        "label"          : "cell_type_L3_refined",
        "name"           : "Myeloid",
        "batch_key_local": "sample",
    },
    "bcell": {
        "h5ad"           : Path("/home/h2048/data/py/0203/bcell_scarches_v4_1/results/scarches_package/"
                                "bcell_reference_20260203.h5ad"),
        "label"          : "cell_type_level_3",
        "name"           : "Bcell",
        "batch_key_local": "sample",
    },
    "stromal": {
        "h5ad"           : Path("/home/h2048/data/py/0308/stromal_reintegration_v1_3/"
                                "stromal_reintegrated_scvi_scanvi_v1_3.h5ad"),
        "label"          : "cell_type_L3",
        "name"           : "Stromal",
        "batch_key_local": "sample",
    },
}

BATCH_KEY                = "sample"
LABELS_KEY               = "scanvi_label"
UNLABELED_CATEGORY       = "Unknown"
N_HVG                    = 4000
N_LATENT                 = 100
SCVI_EPOCHS              = 400
SCANVI_EPOCHS            = 200
ALLOW_UNKNOWN            = False
DROP_UNKNOWN_IF_DISALLOWED = True

OUTPUT_H5AD   = OUTPUT_DIR / f"allcells_combined_{DATE_TAG}_v{VERSION}.h5ad"
PIPELINE_NAME = f"allcells_combined_v{VERSION}"

for key, cfg in LINEAGE_INPUTS.items():
    path = cfg["h5ad"]
    if not path.exists():
        matches = [str(p) for p in Path("/home/h2048/data").rglob(path.name)]
        hint = f" Similar matches: {matches[:5]}" if matches else " No basename matches under /home/h2048/data."
        raise FileNotFoundError(f"[X] {key}: missing input file: {path}.{hint}")

print(f"Output dir : {OUTPUT_DIR}")
print(f"Output h5ad: {OUTPUT_H5AD.name}")
print(f"Batch key  : {BATCH_KEY} (sample-level, unified)")
print(f"Label tier : L3")
print(f"N_HVG      : {N_HVG}  |  N_LATENT: {N_LATENT}")
print(f"ALLOW_UNKNOWN={ALLOW_UNKNOWN} | DROP_UNKNOWN_IF_DISALLOWED={DROP_UNKNOWN_IF_DISALLOWED}")
print("\nVerified lineage inputs:")
for key, cfg in LINEAGE_INPUTS.items():
    print(f"  {key:12s}: label={cfg['label']:<24s} path={cfg['h5ad']}")

## Cell 3 — Helper Functions

In [6]:
_SCVI_INTERNAL_OBS_COLS = {
    "_scvi_batch", "_scvi_labels",
    "_scvi_extra_categorical_covs",
    "_scvi_extra_continuous_covs",
}
_HVG_AUX_VAR_COLS = {
    "highly_variable", "highly_variable_rank", "means", "variances",
    "variances_norm", "dispersions", "dispersions_norm",
    "highly_variable_nbatches",
}


def normalize_lineage_structure(adata_lin, key, local_bk, global_bk):
    """
    [v1.3-1 / v1.3.1-3] Standardize lineage AnnData in-place.
    1. counts layer  : ensure CSR float32; create from raw_counts/.raw if absent
    2. batch column  : unify local_bk -> global_bk
    3. obs cleanup   : remove scVI internal cols; object-backed category dtype
    4. var cleanup   : drop stale HVG aux cols (QRM Sec 12.5)
    5. obsm cleanup  : clear all lineage embeddings
    NOTE: log1p pre-build removed in v1.3.1-3 (not used downstream).
    """
    print(f"  [normalize] {key}")

    # 1. counts layer
    if "counts" not in adata_lin.layers:
        if "raw_counts" in adata_lin.layers:
            adata_lin.layers["counts"] = adata_lin.layers["raw_counts"]
            print(f"    counts: created from layers['raw_counts']")
        elif adata_lin.raw is not None:
            raw_idx = pd.Index(adata_lin.raw.var_names)
            cur_idx = pd.Index(adata_lin.var_names)
            shared  = raw_idx.intersection(cur_idx)
            if len(shared) == len(cur_idx):
                pos = raw_idx.get_indexer(cur_idx)
                adata_lin.layers["counts"] = sparse.csr_matrix(
                    adata_lin.raw.X[:, pos], dtype=np.float32
                )
                print(f"    counts: aligned from .raw.X ({len(shared)} genes)")
            else:
                raise ValueError(
                    f"[X] {key}: counts absent; .raw covers {len(shared)}/{len(cur_idx)} genes"
                )
        else:
            raise ValueError(
                f"[X] {key}: no counts source. Layers: {list(adata_lin.layers.keys())}"
            )

    if not sparse.isspmatrix_csr(adata_lin.layers["counts"]):
        adata_lin.layers["counts"] = sparse.csr_matrix(
            adata_lin.layers["counts"], dtype=np.float32)
    elif adata_lin.layers["counts"].dtype != np.float32:
        adata_lin.layers["counts"] = adata_lin.layers["counts"].astype(np.float32)

    # Global finite/negative check on .data array (QRM Sec 16.1)
    _d = adata_lin.layers["counts"].data
    if len(_d) > 0:
        if not np.isfinite(_d).all():
            raise ValueError(f"[X] {key}: non-finite values in counts")
        if np.any(_d < 0):
            raise ValueError(f"[X] {key}: negative values in counts")
    print(f"    counts: CSR float32 {adata_lin.layers['counts'].shape}")

    # 2. batch column
    if global_bk not in adata_lin.obs.columns:
        if local_bk in adata_lin.obs.columns:
            adata_lin.obs[global_bk] = adata_lin.obs[local_bk].astype(object)
            print(f"    batch: '{local_bk}' -> '{global_bk}' "
                  f"({adata_lin.obs[global_bk].nunique()} levels)")
        else:
            cands = [col for col in adata_lin.obs.columns
                     if any(k in col.lower() for k in ("sample", "batch", "dataset"))]
            raise ValueError(
                f"[X] {key}: batch col '{local_bk}'/'{global_bk}' not found. "
                f"Candidates: {cands}"
            )
    else:
        adata_lin.obs[global_bk] = adata_lin.obs[global_bk].astype(object)
        print(f"    batch: '{global_bk}' present "
              f"({adata_lin.obs[global_bk].nunique()} levels)")

    # 3. obs cleanup
    drop_obs = [col for col in adata_lin.obs.columns if col in _SCVI_INTERNAL_OBS_COLS]
    if drop_obs:
        adata_lin.obs.drop(columns=drop_obs, inplace=True)
        print(f"    obs: dropped internal scVI cols: {drop_obs}")
    for col in adata_lin.obs.select_dtypes(include=["category"]).columns:
        cats = adata_lin.obs[col].cat.categories
        if hasattr(cats.dtype, "name") and cats.dtype.name in ("string", "StringDtype"):
            adata_lin.obs[col] = adata_lin.obs[col].cat.rename_categories(
                cats.astype(object))

    # 4. var cleanup
    drop_var = [col for col in adata_lin.var.columns if col in _HVG_AUX_VAR_COLS]
    if drop_var:
        adata_lin.var.drop(columns=drop_var, inplace=True)
        print(f"    var: dropped HVG aux cols: {drop_var}")

    # 5. obsm cleanup
    n_obsm = len(adata_lin.obsm)
    if n_obsm:
        adata_lin.obsm.clear()
        print(f"    obsm: cleared {n_obsm} lineage embeddings")


def resolve_fullgene_counts_matrix(adata_lin, key):
    """
    [v1.3.1-2/5] Select counts source with most genes; return CSR float32.
    Priority (most genes first; raw > counts > raw_counts on tie).
    Checks: non-finite, negative. Soft integer warning (no hard fail).
    """
    candidates = []
    if adata_lin.raw is not None:
        candidates.append(dict(source="adata.raw.X", X=adata_lin.raw.X,
                               var=adata_lin.raw.var.copy(),
                               n_vars=adata_lin.raw.n_vars, rank=0))
    if "counts" in adata_lin.layers:
        candidates.append(dict(source="layers['counts']", X=adata_lin.layers["counts"],
                               var=adata_lin.var.copy(),
                               n_vars=adata_lin.n_vars, rank=1))
    if "raw_counts" in adata_lin.layers:
        candidates.append(dict(source="layers['raw_counts']", X=adata_lin.layers["raw_counts"],
                               var=adata_lin.var.copy(),
                               n_vars=adata_lin.n_vars, rank=2))
    if not candidates:
        raise ValueError(
            f"[X] {key}: no counts source. "
            f"Layers: {list(adata_lin.layers.keys())} | .raw: {adata_lin.raw is not None}"
        )
    candidates.sort(key=lambda c: (-c["n_vars"], c["rank"]))

    skipped = []
    for cand in candidates:
        X, src = cand["X"], cand["source"]
        _d = X.data if sparse.issparse(X) else np.asarray(X).ravel()
        if len(_d) > 0 and not np.isfinite(_d).all():
            skipped.append(f"    [SKIP] {src}: non-finite"); continue
        if len(_d) > 0 and np.any(_d < 0):
            skipped.append(f"    [SKIP] {src}: negative values"); continue

        # [v1.3.1-5] soft integer warning
        if len(_d) > 0:
            _s = _d[:min(50000, len(_d))]
            frac = np.abs(_s - np.round(_s))
            if np.nanmax(frac) > 1e-3:
                print(f"  [WARNING] {key}: {src} has non-integer values "
                      f"(max_frac={np.nanmax(frac):.4f}) -- may be normalised")

        if skipped:
            print(f"  [INFO] {key}: skipped candidates:")
            for msg in skipped:
                print(msg)
        if src != "adata.raw.X" and adata_lin.raw is not None:
            print(f"  [WARNING] {key}: .raw ({adata_lin.raw.n_vars:,} genes) not used; "
                  f"using {src} ({cand['n_vars']:,} genes)")

        # [v1.3.1-2] Normalise to CSR float32
        if sparse.issparse(X):
            if not sparse.isspmatrix_csr(X):
                X = X.tocsr()
            if X.dtype != np.float32:
                X = X.astype(np.float32)
        else:
            X = sparse.csr_matrix(np.asarray(X, dtype=np.float32))

        print(f"  [OK] {key}: {src} | genes={cand['n_vars']:,}")
        return X, src, cand["var"]

    raise ValueError(
        f"[X] {key}: all candidates failed. " + " | ".join(skipped)
    )


def _safe_label_column(adata_lin, cfg_label, key):
    """
    [v1.3.1-4] Zero silent fallback. Only B cell allows Cell_Type_L3->L2.
    All other lineages: raise ValueError if cfg_label absent.
    """
    if cfg_label in adata_lin.obs.columns:
        return cfg_label
    if key == "bcell" and cfg_label == "Cell_Type_L3":
        if "Cell_Type_L2" in adata_lin.obs.columns:
            print(f"  [WARNING] bcell: 'Cell_Type_L3' not found; "
                  f"using 'Cell_Type_L2' (documented exception)")
            return "Cell_Type_L2"
    raise ValueError(
        f"[X] {key}: label column '{cfg_label}' not found.\n"
        f"    L3-level annotation required.\n"
        f"    Available: {sorted(adata_lin.obs.columns.tolist())}"
    )


def _fmt_float(val, fallback="not_set"):
    return f"{val:.4f}" if isinstance(val, (float, np.floating)) else str(fallback)


print("[OK] Helper functions defined")

## Cell 4 — Load Lineages + Normalize

In [21]:
print("\n" + "="*70)
print("[STEP 4] Loading and normalizing lineage objects")
print("="*70)

lineage_adatas = []
label_sets     = {}
gene_sets      = {}

for key, cfg in LINEAGE_INPUTS.items():
    print(f"\n[{key}] {cfg['h5ad'].name}")
    adata_lin = sc.read_h5ad(cfg["h5ad"])
    print(f"  shape : {adata_lin.n_obs:,} x {adata_lin.n_vars:,} | "
          f".raw : {adata_lin.raw.n_vars if adata_lin.raw is not None else 'None'} genes")

    normalize_lineage_structure(
        adata_lin,
        key,
        local_bk  = cfg.get("batch_key_local", BATCH_KEY),
        global_bk = BATCH_KEY,
    )

    counts_X, count_source, var_df = resolve_fullgene_counts_matrix(adata_lin, key)
    label_col = _safe_label_column(adata_lin, cfg["label"], key)

    adata_full = sc.AnnData(
        X   = counts_X,
        obs = adata_lin.obs.copy(),
        var = var_df,
    )
    adata_full.layers["counts"]          = adata_full.X   # shared ref, no copy
    adata_full.obs["lineage_source"]     = cfg["name"]
    adata_full.obs["cell_type_original"] = (
        adata_lin.obs[label_col].astype(object).fillna(UNLABELED_CATEGORY)
    )

    label_sets[key] = set(adata_full.obs["cell_type_original"].unique())
    gene_sets[key]  = set(adata_full.var_names.tolist())

    print(f"  -> {adata_full.n_obs:,} x {adata_full.n_vars:,} | "
          f"{len(label_sets[key])} L3 classes | src='{count_source}' | col='{label_col}'")

    lineage_adatas.append(adata_full)
    del adata_lin
    gc.collect()

print("\n[OK] All lineages loaded")

## Cell 5 — Pre-concat Audits (label overlap + gene universe)

In [22]:
print("\n" + "="*70)
print("[STEP 5] Pre-concat audits")
print("="*70)

# Label overlap
print("\n[5a] Cross-lineage L3 label overlap")
all_keys = list(label_sets.keys())
for i in range(len(all_keys)):
    for j in range(i + 1, len(all_keys)):
        ka, kb = all_keys[i], all_keys[j]
        overlap = sorted((label_sets[ka] & label_sets[kb]) - {UNLABELED_CATEGORY})
        if overlap:
            print(f"  [INFO] {ka} x {kb}: {len(overlap)} shared -> ONE class in scANVI")
            print(f"         {overlap}")
            print(f"         set USE_LINEAGE_PREFIX=True to separate")
        else:
            print(f"  [OK]   {ka} x {kb}: no overlap")

USE_LINEAGE_PREFIX = False
if USE_LINEAGE_PREFIX:
    for ad, (key, cfg) in zip(lineage_adatas, LINEAGE_INPUTS.items()):
        ad.obs["cell_type_original"] = (
            cfg["name"] + "|" + ad.obs["cell_type_original"].astype(str)
        )
    print("[OK] Lineage prefix applied")

# Gene universe
print("\n[5b] Gene universe consistency")
for k in gene_sets:
    print(f"  {k:12s}: {len(gene_sets[k]):,} genes")
intersection = set.intersection(*gene_sets.values())
union        = set.union(*gene_sets.values())
coverage     = len(intersection) / max(len(union), 1)
print(f"\n  Inner-join : {len(intersection):,} genes")
print(f"  Union      : {len(union):,} genes")
print(f"  Coverage   : {coverage:.3f}")
if len(intersection) < 10000:
    print("  [WARNING] < 10k genes: most lineages have HVG-only counts")
    print("    HVG selection runs on intersection of per-lineage HVG sets")
elif coverage < 0.8:
    print("  [WARNING] Coverage < 0.8 -- check gene naming")

## Cell 6 — Concatenate

In [23]:
print("\n" + "="*70)
print("[STEP 6] Concatenating lineage objects")
print("="*70)

adata = sc.concat(
    lineage_adatas,
    join         = "inner",
    merge        = "unique",
    uns_merge    = "unique",
    label        = "lineage_concat_key",
    keys         = list(LINEAGE_INPUTS.keys()),
    index_unique = "-",
)
del lineage_adatas; gc.collect()

print(f"Combined: {adata.n_obs:,} x {adata.n_vars:,}")
print(f"\nLineage distribution:")
print(adata.obs["lineage_source"].value_counts().to_string())

# Global integrity check on .data array (QRM Sec 16.1)
assert "counts" in adata.layers, "[X] counts missing after concat"
_d = (adata.layers["counts"].data if sparse.issparse(adata.layers["counts"])
      else adata.layers["counts"].ravel())
assert np.isfinite(_d).all(), "[X] Non-finite in counts"
assert np.all(_d >= 0),       "[X] Negative in counts"
del _d; gc.collect()
print("\n[OK] counts integrity verified")

## Cell 7 — Build Unified `scanvi_label` (L3)

In [24]:
print("\n" + "="*70)
print("[STEP 7] Building unified scanvi_label (L3)")
print("="*70)

adata.obs[LABELS_KEY] = pd.Categorical(
    adata.obs["cell_type_original"].astype(object).fillna(UNLABELED_CATEGORY)
)
if UNLABELED_CATEGORY not in adata.obs[LABELS_KEY].cat.categories:
    adata.obs[LABELS_KEY] = adata.obs[LABELS_KEY].cat.add_categories([UNLABELED_CATEGORY])

n_labeled = int((adata.obs[LABELS_KEY] != UNLABELED_CATEGORY).sum())
n_unknown = int((adata.obs[LABELS_KEY] == UNLABELED_CATEGORY).sum())
print(f"Labeled  : {n_labeled:,}  ({n_labeled/adata.n_obs*100:.1f}%)")
print(f"Unknown  : {n_unknown:,}  ({n_unknown/adata.n_obs*100:.1f}%)")
print(f"Distinct : {adata.obs[LABELS_KEY].nunique()} L3 classes")
print(adata.obs[LABELS_KEY].value_counts().to_string())

dropped_unknown = 0
if n_unknown > 0:
    unknown_by_lineage = (
        adata.obs.loc[adata.obs[LABELS_KEY] == UNLABELED_CATEGORY, "lineage_source"]
        .value_counts()
        .sort_values(ascending=False)
    )
    print("\n[INFO] Unknown by lineage:")
    print(unknown_by_lineage.to_string())

if not ALLOW_UNKNOWN and n_unknown > 0:
    if DROP_UNKNOWN_IF_DISALLOWED:
        dropped_unknown = n_unknown
        print(f"\n[WARNING] Dropping {dropped_unknown:,} '{UNLABELED_CATEGORY}' cells for fully supervised training")
        keep_mask = adata.obs[LABELS_KEY] != UNLABELED_CATEGORY
        adata = adata[keep_mask].copy()
        adata.obs[LABELS_KEY] = adata.obs[LABELS_KEY].cat.remove_unused_categories()
        n_labeled = int((adata.obs[LABELS_KEY] != UNLABELED_CATEGORY).sum())
        n_unknown = int((adata.obs[LABELS_KEY] == UNLABELED_CATEGORY).sum())
        print(f"[OK] After drop: {adata.n_obs:,} cells | labeled={n_labeled:,} | unknown={n_unknown:,}")
    else:
        assert n_unknown == 0, (
            f"[X] {n_unknown:,} cells = '{UNLABELED_CATEGORY}'.\n"
            f"    Fix label column, enable semi-supervised mode, or set DROP_UNKNOWN_IF_DISALLOWED = True."
        )
elif ALLOW_UNKNOWN and n_unknown > 0:
    print(f"[INFO] ALLOW_UNKNOWN=True: {n_unknown:,} semi-supervised cells")

adata.obs["cell_type_final"] = adata.obs["cell_type_original"].astype(object)
adata.uns["dropped_unknown_cells"] = int(dropped_unknown)
adata.uns["allow_unknown"] = bool(ALLOW_UNKNOWN)

## Cell 8 — Covariates (MT%)

In [25]:
print("\n" + "="*70)
print("[STEP 8] Covariates")
print("="*70)

mt_mask = adata.var_names.str.startswith("MT-")
n_mt    = int(mt_mask.sum())
_total  = np.array(adata.layers["counts"].sum(axis=1)).ravel().astype(np.float64)
if n_mt > 0:
    _mt = np.array(
        adata.layers["counts"][:, mt_mask].sum(axis=1)
    ).ravel().astype(np.float64)
    adata.obs["pct_counts_mt"] = (_mt / np.maximum(_total, 1) * 100).astype(np.float32)
    del _mt
else:
    adata.obs["pct_counts_mt"] = np.float32(0.0)
    print("[WARNING] No MT- genes")
del _total
print(f"MT genes  : {n_mt}")
print(f"pct_mt    : mean={adata.obs['pct_counts_mt'].mean():.2f}%  "
      f"max={adata.obs['pct_counts_mt'].max():.2f}%")

## Cell 9 — Filter Genes & Cells

In [26]:
print("\n" + "="*70)
print("[STEP 9] Filter genes and cells (QRM Sec 16.4)")
print("="*70)
n_c, n_g = adata.n_obs, adata.n_vars
sc.pp.filter_genes(adata, min_cells=3)
sc.pp.filter_cells(adata, min_genes=200)
print(f"Genes : {n_g:,} -> {adata.n_vars:,}")
print(f"Cells : {n_c:,} -> {adata.n_obs:,}")
small = adata.obs[BATCH_KEY].value_counts()
small = small[small < 3]
if len(small):
    print(f"[WARNING] {len(small)} batches <3 cells: {small.index.tolist()}")

## Cell 10 — HVG Selection (4-tier fallback)

In [27]:
print("\n" + "="*70)
print("[STEP 10] HVG selection (QRM Sec 16.5)")
print("="*70)

# Clear stale HVG aux cols that survived concat (QRM Sec 12.5)
_drop = [c for c in _HVG_AUX_VAR_COLS if c in adata.var.columns]
if _drop:
    adata.var.drop(columns=_drop, inplace=True)

hvg_method = None
try:
    sc.pp.highly_variable_genes(
        adata, layer="counts", n_top_genes=N_HVG,
        batch_key=BATCH_KEY, flavor="seurat_v3", subset=False)
    hvg_method = "seurat_v3 batch-aware (counts)"
except Exception as e1:
    print(f"  Tier1 failed: {e1}")
    try:
        sc.pp.highly_variable_genes(
            adata, layer="counts", n_top_genes=N_HVG,
            flavor="seurat_v3", subset=False)
        hvg_method = "seurat_v3 non-batch (counts)"
    except Exception as e2:
        print(f"  Tier2 failed: {e2}")
        _cnt   = adata.layers["counts"]
        _tot   = np.array(_cnt.sum(axis=1)).ravel().astype(np.float64)
        _scale = sparse.diags(1e4 / np.maximum(_tot, 1))
        _l1p   = (_scale @ _cnt).tocsr()
        _l1p.data = np.log1p(_l1p.data).astype(np.float32)
        adata.layers["_log1p_tmp"] = _l1p
        del _cnt, _tot, _scale, _l1p; gc.collect()
        try:
            sc.pp.highly_variable_genes(
                adata, layer="_log1p_tmp", n_top_genes=N_HVG,
                batch_key=BATCH_KEY, flavor="seurat", subset=False)
            hvg_method = "seurat batch-aware (log1p)"
        except Exception as e3:
            print(f"  Tier3 failed: {e3}")
            sc.pp.highly_variable_genes(
                adata, layer="_log1p_tmp", n_top_genes=N_HVG,
                flavor="seurat", subset=False)
            hvg_method = "seurat non-batch (log1p)"
        del adata.layers["_log1p_tmp"]; gc.collect()

n_hvg = int(adata.var["highly_variable"].sum())
assert n_hvg > 0, "[X] Zero HVGs"
adata.uns["hvg_method"] = hvg_method
print(f"[OK] HVG: {n_hvg:,} | method: {hvg_method}")

pd.Series(adata.var_names[adata.var["highly_variable"]].tolist()).to_csv(
    OUTPUT_DIR / "hvg_genes_final.csv", index=False, header=False)
pd.Series(adata.var_names.tolist()).to_csv(
    OUTPUT_DIR / "all_genes_raw.csv", index=False, header=False)

## Cell 11 — Save Full-Gene `.raw` (shared memory, QRM item 2)

In [28]:
print("\n" + "="*70)
print("[STEP 11] Save full-gene .raw (shared memory)")
print("="*70)
# [QRM 2] no .copy() on counts -- zero extra memory cost
adata.raw = sc.AnnData(
    X   = adata.layers["counts"],
    obs = adata.obs.copy(),
    var = adata.var.copy(),
)
print(f"[OK] .raw: {adata.raw.n_vars:,} genes (shared memory)")

import psutil
print(f"Memory   : {psutil.Process(os.getpid()).memory_info().rss / 1e9:.1f} GB")

## Cell 12 — Subset to HVG

In [29]:
print("\n" + "="*70)
print("[STEP 12] HVG subset")
print("="*70)
adata = adata[:, adata.var["highly_variable"]].copy()
print(f"[OK] {adata.n_obs:,} x {adata.n_vars:,}")
print(f"Memory : {psutil.Process(os.getpid()).memory_info().rss / 1e9:.1f} GB")

## Cell 13 — Normalize `.X` log1p (after HVG subset)

In [30]:
print("\n" + "="*70)
print("[STEP 13] Normalize .X log1p (HVG subset only)")
print("="*70)
adata.X = adata.layers["counts"].copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.layers["log1p"] = adata.X.copy()
print("[OK] .X = log1p (HVG)")

## Cell 14 — scVI Training

In [31]:
print("\n" + "="*70)
print("[STEP 14] scVI training")
print("="*70)

scvi.model.SCVI.setup_anndata(
    adata,
    layer                     = "counts",
    batch_key                 = BATCH_KEY,
    continuous_covariate_keys = ["pct_counts_mt"],
)

model_scvi = scvi.model.SCVI(
    adata,
    n_latent          = N_LATENT,
    n_layers          = 2,
    n_hidden          = 128,
    dropout_rate      = 0.1,
    gene_likelihood   = "nb",
    dispersion        = "gene-batch",
    encode_covariates = True,   # [QRM 15] scArches compatibility
)

# [QRM 7] param count via try-except
try:
    n_params = model_scvi.module.n_params
except AttributeError:
    n_params = sum(p.numel() for p in model_scvi.module.parameters() if p.requires_grad)
print(f"scVI params: {n_params:,}")

model_scvi.train(
    max_epochs     = SCVI_EPOCHS,
    batch_size     = 256,
    train_size     = 0.9,
    early_stopping = True,
    plan_kwargs    = {"lr": 1e-3},
    accelerator    = _accelerator,
    devices        = _devices,
)
adata.obsm["X_scvi"] = model_scvi.get_latent_representation()

SCVI_MODEL_PATH = MODEL_DIR / "scvi_model"
model_scvi.save(str(SCVI_MODEL_PATH), overwrite=True)
# [QRM 17] var_names.csv for scArches
pd.Series(adata.var_names.tolist()).to_csv(
    SCVI_MODEL_PATH / "var_names.csv", index=False, header=False)

# [QRM 16] scArches dry-run: 2k subsample
try:
    _idx = np.random.choice(adata.n_obs, min(2000, adata.n_obs), replace=False)
    _ck  = adata[_idx].copy()
    scvi.model.SCVI.prepare_query_anndata(_ck, str(SCVI_MODEL_PATH))
    del _ck; gc.collect()
    print("[OK] scVI scArches dry-run (2k)")
except Exception as e:
    print(f"[WARNING] scVI dry-run: {e}")

print(f"[OK] scVI saved: {SCVI_MODEL_PATH}")

## Cell 15 — UMAP (scVI latent)

In [32]:
print("\n" + "="*70)
print("[STEP 15] Neighbors + UMAP (scVI)")
print("="*70)
sc.pp.neighbors(adata, use_rep="X_scvi", n_neighbors=20)
sc.tl.umap(adata, min_dist=0.3)
adata.obsm["X_umap_scvi"] = adata.obsm["X_umap"].copy()
print("[OK] X_umap_scvi saved")

## Cell 16 — scANVI Training (L3, fully supervised)

In [34]:
print("\n" + "="*70)
print("[STEP 16] scANVI training (L3 fully supervised)")
print("="*70)

# [P0-5] from_scvi_model inherits registry -- no SCANVI.setup_anndata()
assert LABELS_KEY in adata.obs.columns
if UNLABELED_CATEGORY not in adata.obs[LABELS_KEY].cat.categories:
    adata.obs[LABELS_KEY] = adata.obs[LABELS_KEY].cat.add_categories([UNLABELED_CATEGORY])
    print(f"[INFO] Added unused '{UNLABELED_CATEGORY}' category for SCANVI compatibility")

model_scanvi = scvi.model.SCANVI.from_scvi_model(
    model_scvi,
    unlabeled_category = UNLABELED_CATEGORY,
    labels_key         = LABELS_KEY,
)
model_scanvi.train(
    max_epochs     = SCANVI_EPOCHS,
    batch_size     = 256,
    train_size     = 0.9,
    early_stopping = True,
    plan_kwargs    = {"lr": 1e-3, "weight_decay": 0.0},
    accelerator    = _accelerator,
    devices        = _devices,
)
adata.obsm["X_scanvi"] = model_scanvi.get_latent_representation()

# [QRM 9] label order from predict(soft=True), not hardcoded
soft_df = model_scanvi.predict(soft=True)
adata.obs["cell_type_scanvi_pred_pan"] = soft_df.idxmax(axis=1).values
adata.obs["scanvi_confidence"]         = soft_df.max(axis=1).values.astype(np.float32)
adata.obsm["scanvi_probabilities"]     = soft_df.values.astype(np.float32)
adata.uns["scanvi_celltype_order"]     = soft_df.columns.tolist()

print(f"[OK] scANVI: {soft_df.shape[1]} L3 classes | "
      f"mean confidence={adata.obs['scanvi_confidence'].mean():.4f}")

SCANVI_MODEL_PATH = MODEL_DIR / "scanvi_model"
model_scanvi.save(str(SCANVI_MODEL_PATH), overwrite=True)
# [QRM 17]
pd.Series(adata.var_names.tolist()).to_csv(
    SCANVI_MODEL_PATH / "var_names.csv", index=False, header=False)

# [QRM 16] dry-run 2k
try:
    _idx = np.random.choice(adata.n_obs, min(2000, adata.n_obs), replace=False)
    _ck  = adata[_idx].copy()
    scvi.model.SCANVI.prepare_query_anndata(_ck, str(SCANVI_MODEL_PATH))
    del _ck; gc.collect()
    print("[OK] scANVI scArches dry-run (2k)")
except Exception as e:
    print(f"[WARNING] scANVI dry-run: {e}")

# [QRM 4] release models
del model_scvi, model_scanvi; gc.collect()
if GPU_AVAILABLE:
    torch.cuda.empty_cache()
print(f"[OK] scANVI saved: {SCANVI_MODEL_PATH}")

## Cell 17 — UMAP (scANVI) + Agreement Check

In [35]:
print("\n" + "="*70)
print("[STEP 17] Neighbors + UMAP (scANVI) + agreement check")
print("="*70)
sc.pp.neighbors(adata, use_rep="X_scanvi", n_neighbors=20)
sc.tl.umap(adata, min_dist=0.3)
adata.obsm["X_umap_scanvi"] = adata.obsm["X_umap"].copy()

agree = (
    adata.obs["cell_type_scanvi_pred_pan"].astype(str) ==
    adata.obs["cell_type_final"].astype(str)
)
agreement_rate = float(agree.mean())
adata.uns["agreement_rate_overall"] = agreement_rate
print(f"Overall L3 agreement: {agreement_rate*100:.2f}%")
for lin in adata.obs["lineage_source"].unique():
    ag = agree[adata.obs["lineage_source"] == lin].mean()
    print(f"  {lin:12s}: {ag*100:.2f}%")

## Cell 18 — UMAP Figures

In [36]:
print("\n" + "="*70)
print("[STEP 18] UMAP figures")
print("="*70)

# [QRM v3.6 / Sec 11] vector_friendly=True; no rasterized= kwarg
sc.settings.vector_friendly = True

_color_keys = ["lineage_source", "cell_type_final", BATCH_KEY, "scanvi_confidence"]
for _basis, _lbl in [("X_umap_scvi", "scVI"), ("X_umap_scanvi", "scANVI")]:
    fig, axes = plt.subplots(1, len(_color_keys), figsize=(6 * len(_color_keys), 5))
    for ax, col in zip(axes, _color_keys):
        sc.pl.embedding(
            adata, basis=_basis, color=col, ax=ax, show=False, title=col,
            legend_loc="right margin" if adata.obs[col].nunique() <= 30 else "none",
            frameon=False,
        )
    fig.suptitle(f"{_lbl} UMAP - All Lineages (L3 Frozen)", y=1.02, fontsize=14)
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"allcells_{_lbl.lower()}_umap_overview.pdf",
                dpi=300, bbox_inches="tight")
    plt.close("all")
    print(f"[OK] {_lbl} UMAP saved")

## Cell 19 — Pre-write Cleanup (QRM v3.5 §17.1-17.3)

In [37]:
print("\n" + "="*70)
print("[STEP 19] Pre-write cleanup")
print("="*70)

# [QRM-34] _index reserved column (v3.5 §17.2)
for _attr in ("obs", "var"):
    _df = getattr(adata, _attr)
    if "_index" in _df.columns:
        setattr(adata, _attr, _df.rename(columns={"_index": "orig_index"}))
        print(f"  Renamed {_attr}['_index'] -> 'orig_index'")

# [QRM-35] stale obsm (v3.5 §17.3)
OBSM_KEEP = {"X_scvi","X_scanvi","X_umap_scvi","X_umap_scanvi","scanvi_probabilities"}
stale = [k for k in list(adata.obsm.keys()) if k not in OBSM_KEEP]
for k in stale:
    del adata.obsm[k]
if stale:
    print(f"  Removed stale obsm: {stale}")

# [QRM v3.5 §17.1] object-backed category dtype
for col in adata.obs.select_dtypes(include=["category"]).columns:
    cats = adata.obs[col].cat.categories
    if hasattr(cats.dtype, "name") and cats.dtype.name in ("string","StringDtype"):
        adata.obs[col] = adata.obs[col].cat.rename_categories(cats.astype(object))

print("[OK] Pre-write cleanup done")

## Cell 20 — Reference Manifest JSON (QRM item 19)

In [42]:
print("\n" + "="*70)
print("[STEP 20] reference_manifest.json")
print("="*70)

manifest = {
    "reference_h5ad"      : str(OUTPUT_H5AD),
    "scvi_model_dir"      : str(SCVI_MODEL_PATH),
    "scanvi_model_dir"    : str(SCANVI_MODEL_PATH),
    "hvg_var_names_csv"   : str(SCVI_MODEL_PATH / "var_names.csv"),
    "all_genes_csv"       : str(OUTPUT_DIR / "all_genes_raw.csv"),
    "batch_key"           : BATCH_KEY,
    "batch_granularity"   : "sample",
    "label_key"           : LABELS_KEY,
    "label_tier"          : "L3",
    "final_label_key"     : "cell_type_final",
    "prediction_key"      : "cell_type_scanvi_pred_pan",
    "confidence_key"      : "scanvi_confidence",
    "unlabeled_category"  : UNLABELED_CATEGORY,
    "allow_unknown"       : bool(adata.uns.get("allow_unknown", ALLOW_UNKNOWN)),
    "dropped_unknown_cells": int(adata.uns.get("dropped_unknown_cells", 0)),
    "n_latent"            : N_LATENT,
    "encode_covariates"   : True,
    "scarches_compatible" : True,
    "use_lineage_prefix"  : USE_LINEAGE_PREFIX,
    "agreement_rate"      : agreement_rate,
    "version"             : VERSION,
    "date"                : datetime.now().strftime("%Y-%m-%d"),
    "lineage_inputs"      : {
        k: {"h5ad": str(v["h5ad"]), "label": v["label"], "batch_local": v["batch_key_local"]}
        for k, v in LINEAGE_INPUTS.items()
    },
}
with open(OUTPUT_DIR / "reference_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)
print("[OK] reference_manifest.json written")

## Cell 21 — Save h5ad

In [39]:
print("\n" + "="*70)
print(f"[STEP 21] Saving {OUTPUT_H5AD.name}")
print("="*70)
adata.write_h5ad(OUTPUT_H5AD, compression="gzip", compression_opts=9)
print(f"[OK] Saved: {OUTPUT_H5AD}")
print(f"     Size : {OUTPUT_H5AD.stat().st_size / 1e9:.2f} GB")

## Cell 22 — AnnData Structure JSON Output

In [43]:
print("\n" + "="*70)
print("[STEP 22] AnnData structure JSON output")
print("="*70)

n_labeled_current = int((adata.obs[LABELS_KEY].astype(str) != UNLABELED_CATEGORY).sum())
n_unknown_current = int((adata.obs[LABELS_KEY].astype(str) == UNLABELED_CATEGORY).sum())


def _layer_info(X):
    sp  = sparse.issparse(X)
    fmt = type(X).__name__ if sp else "ndarray"
    return {"type": fmt, "shape": list(X.shape), "dtype": str(X.dtype), "sparse": sp}


def _col_info(series):
    info = {"dtype": str(series.dtype), "non_null": int(series.notna().sum())}
    try: info["unique"] = int(series.nunique())
    except Exception: pass
    return info


structure = {
    "pipeline"             : PIPELINE_NAME,
    "version"              : VERSION,
    "timestamp"            : datetime.now().isoformat(),
    "output_h5ad"          : str(OUTPUT_H5AD),
    "shape"                : [adata.n_obs, adata.n_vars],
    "X"                    : _layer_info(adata.X),
    "raw_n_vars"           : int(adata.raw.n_vars) if adata.raw is not None else None,
    "raw_dtype"            : str(adata.raw.X.dtype) if adata.raw is not None else None,
    "layers"               : {k: _layer_info(v) for k, v in adata.layers.items()},
    "obsm"                 : {
        k: {"shape": list(v.shape), "dtype": str(v.dtype)}
        for k, v in adata.obsm.items()
    },
    "batch_key"            : BATCH_KEY,
    "batch_granularity"    : "sample",
    "n_batches"            : int(adata.obs[BATCH_KEY].nunique()),
    "label_tier"           : "L3",
    "labels_key"           : LABELS_KEY,
    "n_L3_classes"         : int(adata.obs[LABELS_KEY].astype(str).nunique()),
    "n_labeled"            : n_labeled_current,
    "n_unknown"            : n_unknown_current,
    "dropped_unknown_cells": int(adata.uns.get("dropped_unknown_cells", 0)),
    "hvg_method"           : adata.uns.get("hvg_method", "unknown"),
    "n_latent"             : N_LATENT,
    "agreement_rate"       : float(agreement_rate),
    "scanvi_celltype_order": adata.uns.get("scanvi_celltype_order", []),
    "key_obs_columns"      : {
        "cell_type_final"           : _col_info(adata.obs["cell_type_final"]),
        LABELS_KEY                  : _col_info(adata.obs[LABELS_KEY]),
        "cell_type_scanvi_pred_pan" : _col_info(adata.obs["cell_type_scanvi_pred_pan"]),
        "scanvi_confidence"         : _col_info(adata.obs["scanvi_confidence"]),
        "lineage_source"            : _col_info(adata.obs["lineage_source"]),
        BATCH_KEY                   : _col_info(adata.obs[BATCH_KEY]),
    },
    "lineage_distribution" : adata.obs["lineage_source"].value_counts().to_dict(),
    "label_distribution"   : adata.obs["cell_type_final"].value_counts().head(60).to_dict(),
    "confidence_stats"     : {
        k: float(v)
        for k, v in adata.obs["scanvi_confidence"].describe().items()
    },
    "scvi_model_dir"       : str(SCVI_MODEL_PATH),
    "scanvi_model_dir"     : str(SCANVI_MODEL_PATH),
    "log_file"             : str(LOG_FILE),
}

struct_path = OUTPUT_DIR / f"anndata_structure_{DATE_TAG}_v{VERSION}.json"
with open(struct_path, "w") as f:
    json.dump(structure, f, indent=2)
print(f"[OK] Structure JSON: {struct_path}")

# Inline summary
print(f"\n  shape            : {adata.n_obs:,} x {adata.n_vars:,}")
print(f"  .raw             : {adata.raw.n_vars:,} genes")
print(f"  layers           : {list(adata.layers.keys())}")
print(f"  obsm             : {list(adata.obsm.keys())}")
print(f"  batch_key        : {BATCH_KEY} ({structure['n_batches']} samples)")
print(f"  L3 classes       : {structure['n_L3_classes']}")
print(f"  labeled / unknown: {n_labeled_current:,} / {n_unknown_current:,}")
print(f"  agreement_rate   : {agreement_rate*100:.2f}%")

## Cell 23 — Pipeline Run Log + Close Log File

In [44]:
elapsed_min = (time.time() - PIPELINE_START) / 60
n_labeled_current = int((adata.obs[LABELS_KEY].astype(str) != UNLABELED_CATEGORY).sum())
n_unknown_current = int((adata.obs[LABELS_KEY].astype(str) == UNLABELED_CATEGORY).sum())

_log = [
    "=" * 80,
    f"{PIPELINE_NAME} Run Log",
    "=" * 80,
    f"written_at          : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    f"log_file            : {LOG_FILE}",
    f"output_h5ad         : {OUTPUT_H5AD}",
    f"n_cells             : {adata.n_obs:,}",
    f"n_genes_hvg         : {adata.n_vars:,}",
    f"n_genes_full (.raw) : {adata.raw.n_vars if adata.raw is not None else 'None'}",
    f"n_batches           : {adata.obs[BATCH_KEY].nunique():,}",
    f"batch_granularity   : sample",
    f"hvg_method          : {adata.uns.get('hvg_method', 'unknown')}",
    f"n_latent            : {N_LATENT}",
    f"label_tier          : L3",
    f"n_labeled           : {n_labeled_current:,}",
    f"n_unknown           : {n_unknown_current:,}",
    f"dropped_unknown     : {int(adata.uns.get('dropped_unknown_cells', 0)):,}",
    f"agreement_rate      : {_fmt_float(adata.uns.get('agreement_rate_overall'))}",
    f"accelerator         : {_accelerator}",
    f"elapsed_minutes     : {elapsed_min:.2f}",
    "",
    "lineage_source distribution:",
    adata.obs["lineage_source"].value_counts().to_string(),
    "",
    "cell_type_final distribution (top 40):",
    adata.obs["cell_type_final"].value_counts().head(40).to_string(),
    "",
    "scanvi_confidence percentiles:",
    adata.obs["scanvi_confidence"].describe().to_string(),
    "",
    "per-lineage label columns used:",
]
for k, v in LINEAGE_INPUTS.items():
    _log.append(f"  {k:12s}: label='{v['label']}' | batch_local='{v['batch_key_local']}'")

_log_text = "\n".join(_log) + "\n"
(OUTPUT_DIR / "pipeline_run_log.txt").write_text(_log_text, encoding="utf-8")
(OUTPUT_DIR / "anndata_structure.txt").write_text(
    f"See anndata_structure_{DATE_TAG}_v{VERSION}.json\n"
    f"n_obs={adata.n_obs:,} n_vars={adata.n_vars:,}\n",
    encoding="utf-8",
)

print("=" * 70)
print("[DONE] Pipeline complete")
print(f"Elapsed  : {elapsed_min:.2f} min")
print(f"Output   : {OUTPUT_H5AD}")
print(f"Log file : {LOG_FILE}")
print("=" * 70)

# ---- Close Tee and restore stdout/stderr ----
try:
    sys.stdout = sys.__stdout__
    sys.stderr = sys.__stderr__
except Exception:
    pass

try:
    if "_log_fh" in globals() and not getattr(_log_fh, "closed", True):
        _log_fh.flush()
        _log_fh.close()
        print(f"[OK] Log file closed: {LOG_FILE}")
    else:
        print(f"[OK] Log file already closed: {LOG_FILE}")
except Exception as e:
    print(f"[WARNING] Log close: {e}")